In [1]:
# pip install pathlib
# !pip install albumentations
# !pip install -U albumentations
# %pip install albumentations
# !{sys.executable} -m pip install albumentations
# pip uninstall -y opencv-python opencv-python-headless
# pip install albumentations
# %pip uninstall -y opencv-python opencv-python-headless
# %pip install albumentations


In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader, Dataset, WeightedRandomSampler

import torchvision
from torchvision.transforms import v2 

from PIL import Image
from pathlib import Path
import cv2

In [3]:
# class_dir = sorted(d for d in root.iterdir() if d.is_dir())
# print(len(class_dir))
# print(class_dir[0])

In [4]:
# first_image = list(class_dir[0].glob("*.JPG"))
# print(len(first_image))
# print(first_image[0].name)

In [5]:
# first_image[0].stem
# first_image[0].stem.split("___")
# first_image[0].stem.split("___")[-1]


In [6]:
# for label, clas in enumerate(class_dir):
    # print(label)
    # print(clas.name,"\n")

In [7]:
root = Path("archive/plantvillage dataset/color")
print(root.exists())

True


### Custom Dataset

In [8]:
class Plant_Dataset(Dataset):
    def __init__(self, base_dir, varient="color", transforms = None):

        self.root = Path(base_dir)/varient
        self.classes = sorted(d.name for d in self.root.iterdir() if d.is_dir())
        
        self.transforms = transforms

        self.samples = []
        for label, class_name in enumerate(self.classes):
            for path in sorted((self.root/class_name).iterdir()):
                if path.suffix.lower() in {".jpg",".jpeg",".png"}:
                    group = path.stem.split("___")[-1]
                    self.samples.append((path, label, group))

        self.target = [s[1] for s in self.samples]
        self.group = [s[2] for s in self.samples]
        
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):

        path, label, _ =  self.samples[index]
        image = cv2.imread(str(path))
        
        if image is None:
            raise ValueError(f"could not read image {path}")
            
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms is not None:
            image = self.transforms(image=image)["image"]

        return image, label

### Transform

In [9]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_tmsf = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Affine(translate_percent=(-0.05, 0.05), scale=(0.9, 1.1), rotate=(-15, 15), p=0.5),
    # A.ShiftScaleRotate(shift_limit = 0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    # A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3), 
    A.Normalize((0.5,0.5,0.5), (0.5, 0.5, 0.5)),
    ToTensorV2()
])

val_tmsf = A.Compose([
    A.Resize(256,256),
    A.Normalize((0.5,0.5, 0.5), (0.5,0.5,0.5)),
    ToTensorV2()
])


In [10]:
train_base = Plant_Dataset("archive/plantvillage dataset", varient = "color", transforms = train_tmsf)
eval_base = Plant_Dataset("archive/plantvillage dataset", varient = "color",  transforms = val_tmsf)
assert train_base.samples == eval_base.samples

In [11]:
len(train_base.classes)

38

In [12]:
# fig, axes = plt.subplots(1,4, figsize= (12,3))

# for ax, i in zip(axes,[0, 500, 6000, 50000]):
#     image, label = train_base[i]
#     ax.imshow(image)
#     ax.set_title(train_base.classes[label], fontsize = 7)
#     ax.axis("off")
# plt.show()    

In [13]:
label = np.array(train_base.target)
group = np.array(train_base.group)
idx = np.arange(len(label))

def hold_out_20(indices):
    skgf = StratifiedGroupKFold(n_splits = 5, shuffle = True, random_state = 42)
    rest, held = next(skgf.split(indices, label[indices], group[indices]))
    return indices[rest] , indices[held]

In [14]:
train_val_idx, test_idx = hold_out_20(idx)
train_idx, val_idx = hold_out_20(train_val_idx)

In [15]:
train_set = Subset(train_base, train_idx)
val_set = Subset(eval_base, val_idx)
test_set = Subset(eval_base, test_idx)

In [16]:
classes = train_base.classes
train_label = label[train_idx]
class_count = np.bincount(train_label, minlength = len(classes))

for i in np.argsort(class_count):
    print(f"{classes[i]} : 50s {class_count[i]}")

print(f"imabalance ratio: {class_count.max() / class_count.min()}")

assert class_count.min() > 0

Potato___healthy : 50s 97
Apple___Cedar_apple_rust : 50s 175
Peach___healthy : 50s 228
Raspberry___healthy : 50s 236
Tomato___Tomato_mosaic_virus : 50s 239
Grape___healthy : 50s 271
Strawberry___healthy : 50s 293
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot : 50s 329
Apple___Black_rot : 50s 398
Apple___Apple_scab : 50s 405
Cherry_(including_sour)___healthy : 50s 544
Tomato___Leaf_Mold : 50s 609
Corn_(maize)___Northern_Leaf_Blight : 50s 632
Tomato___Early_blight : 50s 639
Pepper,_bell___Bacterial_spot : 50s 639
Potato___Early_blight : 50s 640
Potato___Late_blight : 50s 641
Cherry_(including_sour)___Powdery_mildew : 50s 673
Grape___Leaf_blight_(Isariopsis_Leaf_Spot) : 50s 691
Strawberry___Leaf_scorch : 50s 709
Corn_(maize)___healthy : 50s 745
Grape___Black_rot : 50s 756
Corn_(maize)___Common_rust_ : 50s 763
Grape___Esca_(Black_Measles) : 50s 884
Tomato___Target_Spot : 50s 898
Pepper,_bell___healthy : 50s 946
Blueberry___healthy : 50s 966
Tomato___healthy : 50s 1017
Apple___healthy 

### index (paths, labels, groups) → StratifiedGroupKFold split → Subsets → DataLoaders

In [17]:

class_weight = 1.0 / class_count
sample_weight = class_weight[train_label]

generator = torch.Generator().manual_seed(42)

sampler = WeightedRandomSampler(
    weights = torch.as_tensor(sample_weight, dtype = torch.double),
    num_samples = len(train_idx),
    replacement = True,
    generator = generator
)

In [18]:
train_loader = DataLoader(train_set, batch_size = 32, sampler = sampler)
val_loader = DataLoader(val_set, batch_size = 32, shuffle = False)
test_loader = DataLoader(test_set, batch_size =32, shuffle = False)

### Model Building

In [19]:
class Plant_Village(nn.Module):
    def __init__(self, num_classes = 30):
        super().__init__()
        self.num_classes = num_classes

        self.plant_village_cnn = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32, 64, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(128, 256, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
          )

        self.fc_layer = nn.Sequential(
            nn.Linear(16*16*256, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, self.num_classes)
        )


    def forward(self, x):
            
        x = self.plant_village_cnn(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layer(x)
        return x
        

In [20]:
model = Plant_Village(num_classes = len(train_base.classes))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training

In [ ]:
epochs = 15


for epoch in range(epochs):

    training_loss = 0.0
    validation_loss = 0.0

    model.train()

    for image, label in train_loader:

        optimizer.zero_grad()
        
        output = model(image)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()

        training_loss += loss.item()

    avg_train_loss = training_loss / len(train_loader)

    model.eval()
    for image, label in val_loader:

        with torch.no_grad():

            output = model(image)

            loss = criterion(output, label)

            validation_loss += loss.item() 

    avg_val_loss = validation_loss / len(val_loader)

    print(f"Epoch: {epoch+1}/{epochs}, Average Training Loss: {avg_train_loss *100} & Average Validation Loss: {avg_val_loss *100}")    



### Save and Load the best model


In [ ]:


best_val_loss = float("inf")

if avg_val_loss < best_val_loss:

    best_val_loss = avg_val_loss
    torch.save(model.state_dict(), "plant_village_model.pt")




<All keys matched successfully>

In [21]:
model.load_state_dict(torch.load("plant_village_model.pt"))

<All keys matched successfully>

### model testing

In [23]:
correct_label = 0
total_label = 0

model.eval()

with torch.no_grad():
    for image, label in test_loader:

        output = model(image)

        loss = criterion(output, label)

        _, pred = torch.max(output, 1)

        correct_label += (pred == label).sum().item()

        total_label += label.size(0)

        accuracy = correct_label/total_label

print(f"Accuracy: {accuracy *100}%")  

Accuracy: 95.04421518054532%
